# Notebook 03: Predictive Risk Model for Player Performance Decline

**Objective**: Build a trainer-facing logistic regression model to predict player performance decline risk under short rest conditions.

**Input**: Scored outfield performance dataset from Notebook 02 (`arsenal_outfield_performance_scored_min30.csv`)

**Output**: Risk table with player-match decline probability predictions ready for coaching staff.

## 0. Notebook Aim and Modelling Rules

### Primary Goal
Predict player performance decline probability using only **pre-match information** available before kickoff.

### Critical No-Leakage Rule
**Allowed predictors** (information available before match):
- `rest_days`: Days since last appearance
- `short_rest_binary`: Binary indicator (≤4 days = 1)
- `minutes_last_1`, `minutes_last_3`, `minutes_last_5`: Cumulative playing time
- `starts_last_3`, `starts_last_5`: Number of starts in last N matches
- `short_rest_count_last_3`, `short_rest_count_last_5`: Frequency of short-rest appearances
- `champions_league_minutes_last_3`: Cup competition load
- `played_champions_league_last_match`: Binary competition exposure
- `position_group`: Tactical role
- `venue`: Home/Away designation
- `competition`: Match type (league, cup, etc.)
- `previous_overall_score`: Player's last match performance
- `previous_decline_flag`: Prior decline history
- `rolling_player_baseline`: Player's 5-match average (rolling estimate)

### NOT Allowed (Same-Match Outcomes)
- `overall_score`, `attacking_score_normalized`, `defensive_score_normalized`, `discipline_score_normalized`
- `goals`, `assists`, `shots` from current match
- `decline_risk` from current match

These are **outcomes**, not predictors.

### Model Type
Logistic regression: interpretable, trainer-friendly, coefficients show risk direction and magnitude.

In [8]:
# ============================================================================
# NOTEBOOK 03: PREDICTIVE RISK MODEL FOR PERFORMANCE DECLINE
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from datetime import datetime, timedelta

from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve
)

import warnings
warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------------
# Visualisation settings
# ----------------------------------------------------------------------------

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")

# ----------------------------------------------------------------------------
# Team / season configuration
# ----------------------------------------------------------------------------

SEASON = "2024_2025"
TEAM_NAME = "Arsenal"
TEAM_SLUG = "arsenal"

DATA_BASE = Path("/home/vant/Documentos/FOOTBALL_ANALYTICS/MASTER/PROYECT/Data") / f"SEASON_{SEASON}"
TEAM_PATH = DATA_BASE / f"{TEAM_SLUG}_{SEASON}"

PROCESSED_PATH = TEAM_PATH / "processed"
OUTPUTS_PATH = TEAM_PATH / "outputs"
FIGURES_PATH = OUTPUTS_PATH / "figures" / "decline_risk_model"
MODEL_OUTPUT_PATH = OUTPUTS_PATH / "model_outputs"

FIGURES_PATH.mkdir(parents=True, exist_ok=True)
MODEL_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("NOTEBOOK 03: PREDICTIVE RISK MODEL FOR PERFORMANCE DECLINE")
print("=" * 100)

print(f"Team:              {TEAM_NAME}")
print(f"Season:            {SEASON}")
print(f"Data base:         {DATA_BASE}")
print(f"Team path:         {TEAM_PATH}")
print(f"Processed data:    {PROCESSED_PATH}")
print(f"Output figures:    {FIGURES_PATH}")
print(f"Model outputs:     {MODEL_OUTPUT_PATH}")

NOTEBOOK 03: PREDICTIVE RISK MODEL FOR PERFORMANCE DECLINE
Team:              Arsenal
Season:            2024_2025
Data base:         /home/vant/Documentos/FOOTBALL_ANALYTICS/MASTER/PROYECT/Data/SEASON_2024_2025
Team path:         /home/vant/Documentos/FOOTBALL_ANALYTICS/MASTER/PROYECT/Data/SEASON_2024_2025/arsenal_2024_2025
Processed data:    /home/vant/Documentos/FOOTBALL_ANALYTICS/MASTER/PROYECT/Data/SEASON_2024_2025/arsenal_2024_2025/processed
Output figures:    /home/vant/Documentos/FOOTBALL_ANALYTICS/MASTER/PROYECT/Data/SEASON_2024_2025/arsenal_2024_2025/outputs/figures/decline_risk_model
Model outputs:     /home/vant/Documentos/FOOTBALL_ANALYTICS/MASTER/PROYECT/Data/SEASON_2024_2025/arsenal_2024_2025/outputs/model_outputs


## 1. Load Scored Outfield Dataset from Notebook 02

In [9]:
# ============================================================================
# 1. LOAD PERFORMANCE-SCORED DATA FROM NOTEBOOK 02
# ============================================================================

model_input_file = PROCESSED_PATH / f"{TEAM_SLUG}_outfield_performance_scored_min30.csv"

df_model = pd.read_csv(model_input_file)

# Ensure date column is datetime
df_model["date"] = pd.to_datetime(df_model["date"], errors="coerce")

# Sort chronologically
df_model = df_model.sort_values(["date", "match_id", "player"]).reset_index(drop=True)

print("=" * 100)
print("1. LOAD PERFORMANCE-SCORED DATA")
print("=" * 100)

print(f"Input file: {model_input_file}")
print(f"Dataset loaded: {df_model.shape[0]} rows × {df_model.shape[1]} columns")
print(f"\nDate range: {df_model['date'].min()} to {df_model['date'].max()}")
print(f"Unique matches: {df_model['match_id'].nunique()}")
print(f"Unique players: {df_model['player'].nunique()}")

# Required columns for modelling
required_cols = [
    "match_id",
    "date",
    "player",
    "position_group",
    "competition",
    "venue",
    "opponent",
    "min",
    "rest_days",
    "rest_category",
    "overall_score"
]

print("\nRequired columns check:")
for col in required_cols:
    status = "✓" if col in df_model.columns else "✗"
    print(f"  {status} {col}")

print("\nScore columns available:")
score_cols = [
    col for col in df_model.columns
    if "score" in col.lower()
]
print(f"  {score_cols}")

print("\nFirst 5 rows:")
preview_cols = [
    "date", "match_id", "player", "position_group",
    "competition", "venue", "opponent",
    "min", "rest_days", "rest_category", "overall_score"
]
preview_cols = [c for c in preview_cols if c in df_model.columns]
print(df_model[preview_cols].head().to_string(index=False))

1. LOAD PERFORMANCE-SCORED DATA
Input file: /home/vant/Documentos/FOOTBALL_ANALYTICS/MASTER/PROYECT/Data/SEASON_2024_2025/arsenal_2024_2025/processed/arsenal_outfield_performance_scored_min30.csv
Dataset loaded: 629 rows × 62 columns

Date range: 2024-08-17 00:00:00 to 2025-05-25 00:00:00
Unique matches: 58
Unique players: 22

Required columns check:
  ✓ match_id
  ✓ date
  ✓ player
  ✓ position_group
  ✓ competition
  ✓ venue
  ✓ opponent
  ✓ min
  ✓ rest_days
  ✓ rest_category
  ✓ overall_score

Score columns available:
  ['attacking_score', 'attacking_score_normalized', 'defensive_score', 'defensive_score_normalized', 'discipline_score_normalized', 'overall_score', 'attacking_score_raw', 'defensive_score_raw']

First 5 rows:
      date match_id             player     position_group    competition venue opponent  min  rest_days    rest_category  overall_score
2024-08-17 c0e3342a          Ben White          Full-back Premier League  Home   Wolves   90        NaN First appearance      

## 2. Define Player-Relative Decline Target

In [11]:
# ============================================================================
# 2. DEFINE PLAYER-RELATIVE DECLINE TARGET
# ============================================================================

print("\n" + "=" * 100)
print("2. DEFINE PLAYER-RELATIVE DECLINE TARGET")
print("=" * 100)

# Sort by player and date
df_model = df_model.sort_values(["player", "date", "match_id"]).copy()

# Calculate player rolling baseline using ONLY previous matches
df_model["rolling_baseline_5"] = (
    df_model
    .groupby("player")["overall_score"]
    .transform(lambda x: x.shift(1).rolling(window=5, min_periods=2).mean())
)

# Percentage difference from player baseline
df_model["pct_vs_baseline_5"] = (
    (df_model["overall_score"] - df_model["rolling_baseline_5"]) /
    df_model["rolling_baseline_5"]
)

# Define decline target: performance drops >10% below player baseline
df_model["decline_target"] = (
    df_model["pct_vs_baseline_5"] < -0.10
).astype(int)

# Keep only rows with valid baseline
target_df = df_model.dropna(subset=["rolling_baseline_5"]).copy()

print("DECLINE TARGET DEFINITION")
print("  Baseline: player's previous 5-match average overall_score")
print("  If: (current score - baseline) / baseline < -0.10")
print("  Then: decline_target = 1")

print(f"\nRows with valid baseline: {len(target_df)} / {len(df_model)}")
print(f"Rows removed due to insufficient prior matches: {len(df_model) - len(target_df)}")

print("\nTarget distribution:")
print(target_df["decline_target"].value_counts().sort_index().to_string())

print(f"\nDecline rate: {target_df['decline_target'].mean():.1%}")

print("\nExample interpretation:")
print("  Baseline = 70")
print("  10% below baseline = 63")
print("  Score 65 → No decline")
print("  Score 63 → Borderline")
print("  Score 62 → Decline")

print("\nPreview:")
preview_cols = [
    "player", "date", "opponent", "overall_score",
    "rolling_baseline_5", "pct_vs_baseline_5",
    "decline_target", "rest_days", "rest_category"
]

print(
    target_df[preview_cols]
    .head(15)
    .round(3)
    .to_string(index=False)
)


2. DEFINE PLAYER-RELATIVE DECLINE TARGET
DECLINE TARGET DEFINITION
  Baseline: player's previous 5-match average overall_score
  If: (current score - baseline) / baseline < -0.10
  Then: decline_target = 1

Rows with valid baseline: 586 / 629
Rows removed due to insufficient prior matches: 43

Target distribution:
decline_target
0    363
1    223

Decline rate: 38.1%

Example interpretation:
  Baseline = 70
  10% below baseline = 63
  Score 65 → No decline
  Score 63 → Borderline
  Score 62 → Decline

Preview:
   player       date           opponent  overall_score  rolling_baseline_5  pct_vs_baseline_5  decline_target  rest_days rest_category
Ben White 2024-08-31           Brighton         69.329              33.905              1.045               0        7.0   Normal rest
Ben White 2024-09-15  Tottenham Hotspur         50.902              45.713              0.114               0       15.0   Normal rest
Ben White 2024-09-19         itAtalanta         35.409              47.010    

## 3. Engineer Pre-Match Workload Features

In [16]:
# ============================================================================
# 3. CREATE PRE-MATCH WORKLOAD FEATURES
# ============================================================================
# All features are shifted before rolling, so the current match is not used.

print("\n" + "=" * 100)
print("3. CREATE PRE-MATCH WORKLOAD FEATURES")
print("=" * 100)

df_model = df_model.sort_values(["player", "date", "match_id"]).copy()

# ---------------------------------------------------------------------------
# Basic current pre-match rest features
# ---------------------------------------------------------------------------

df_model["short_rest_binary"] = (
    df_model["rest_category"] == "Short rest"
).astype(int)

df_model["moderate_rest_binary"] = (
    df_model["rest_category"] == "Moderate rest"
).astype(int)

df_model["normal_rest_binary"] = (
    df_model["rest_category"] == "Normal rest"
).astype(int)

df_model["long_gap_return"] = (
    df_model["rest_days"] >= 21
).astype(int)

# ---------------------------------------------------------------------------
# Previous minutes workload
# ---------------------------------------------------------------------------

df_model["minutes_last_1"] = (
    df_model
    .groupby("player")["min"]
    .shift(1)
)

for window in [3, 5]:
    df_model[f"minutes_last_{window}"] = (
        df_model
        .groupby("player")["min"]
        .transform(lambda x: x.shift(1).rolling(window=window, min_periods=1).sum())
    )

    df_model[f"avg_minutes_last_{window}"] = (
        df_model
        .groupby("player")["min"]
        .transform(lambda x: x.shift(1).rolling(window=window, min_periods=1).mean())
    )

# ---------------------------------------------------------------------------
# Previous starts workload
# ---------------------------------------------------------------------------

df_model["is_starter"] = (df_model["min"] >= 60).astype(int)

for window in [3, 5]:
    df_model[f"starts_last_{window}"] = (
        df_model
        .groupby("player")["is_starter"]
        .transform(lambda x: x.shift(1).rolling(window=window, min_periods=1).sum())
    )

# ---------------------------------------------------------------------------
# Previous short-rest exposure
# ---------------------------------------------------------------------------

for window in [3, 5]:
    df_model[f"short_rest_count_last_{window}"] = (
        df_model
        .groupby("player")["short_rest_binary"]
        .transform(lambda x: x.shift(1).rolling(window=window, min_periods=1).sum())
    )

# ---------------------------------------------------------------------------
# Previous performance and previous decline history
# ---------------------------------------------------------------------------

df_model["overall_score_last_1"] = (
    df_model
    .groupby("player")["overall_score"]
    .shift(1)
)

for window in [3, 5]:
    df_model[f"overall_score_avg_last_{window}"] = (
        df_model
        .groupby("player")["overall_score"]
        .transform(lambda x: x.shift(1).rolling(window=window, min_periods=1).mean())
    )

df_model["decline_last_1"] = (
    df_model
    .groupby("player")["decline_target"]
    .shift(1)
)

for window in [3, 5]:
    df_model[f"declines_last_{window}"] = (
        df_model
        .groupby("player")["decline_target"]
        .transform(lambda x: x.shift(1).rolling(window=window, min_periods=1).sum())
    )

# ---------------------------------------------------------------------------
# Fill missing previous-history values
# ---------------------------------------------------------------------------

history_cols = [
    "minutes_last_1",
    "minutes_last_3",
    "minutes_last_5",
    "avg_minutes_last_3",
    "avg_minutes_last_5",
    "starts_last_3",
    "starts_last_5",
    "short_rest_count_last_3",
    "short_rest_count_last_5",
    "overall_score_last_1",
    "overall_score_avg_last_3",
    "overall_score_avg_last_5",
    "decline_last_1",
    "declines_last_3",
    "declines_last_5",
]

df_model[history_cols] = df_model[history_cols].fillna(0)

print("✓ Pre-match workload features created")

print("\nFeature groups created:")
print("  ✓ Rest context: rest_days, short_rest_binary, moderate_rest_binary, normal_rest_binary, long_gap_return")
print("  ✓ Recent minutes: minutes_last_1, minutes_last_3, minutes_last_5")
print("  ✓ Recent starts: starts_last_3, starts_last_5")
print("  ✓ Repeated congestion: short_rest_count_last_3, short_rest_count_last_5")
print("  ✓ Recent performance: overall_score_last_1, overall_score_avg_last_3, overall_score_avg_last_5")
print("  ✓ Previous decline history: decline_last_1, declines_last_3, declines_last_5")

print("\nMissingness after fill:")
for col in history_cols:
    print(f"  {col:32} {df_model[col].isna().sum()} missing")

print("\nSample engineered features:")

sample_cols = [
    "player",
    "date",
    "opponent",
    "rest_days",
    "short_rest_binary",
    "min",
    "minutes_last_3",
    "starts_last_3",
    "short_rest_count_last_3",
    "overall_score",
    "rolling_baseline_5",
    "overall_score_last_1",
    "decline_target",
    "declines_last_3",
]

sample_cols = [col for col in sample_cols if col in df_model.columns]

preview_df = (
    df_model
    .dropna(subset=["rolling_baseline_5"])
    [sample_cols]
    .head(15)
)

print(preview_df.round(2).to_string(index=False))


3. CREATE PRE-MATCH WORKLOAD FEATURES
✓ Pre-match workload features created

Feature groups created:
  ✓ Rest context: rest_days, short_rest_binary, moderate_rest_binary, normal_rest_binary, long_gap_return
  ✓ Recent minutes: minutes_last_1, minutes_last_3, minutes_last_5
  ✓ Recent starts: starts_last_3, starts_last_5
  ✓ Repeated congestion: short_rest_count_last_3, short_rest_count_last_5
  ✓ Recent performance: overall_score_last_1, overall_score_avg_last_3, overall_score_avg_last_5
  ✓ Previous decline history: decline_last_1, declines_last_3, declines_last_5

Missingness after fill:
  minutes_last_1                   0 missing
  minutes_last_3                   0 missing
  minutes_last_5                   0 missing
  avg_minutes_last_3               0 missing
  avg_minutes_last_5               0 missing
  starts_last_3                    0 missing
  starts_last_5                    0 missing
  short_rest_count_last_3          0 missing
  short_rest_count_last_5          0 missi

## 4. Add Competition and Champions League Context

In [17]:
# ============================================================================
# 4. CREATE COMPETITION AND CHAMPIONS LEAGUE CONTEXT FEATURES
# ============================================================================
# All features are known before the match or based only on previous matches.

print("\n" + "=" * 100)
print("4. CREATE COMPETITION AND CHAMPIONS LEAGUE CONTEXT FEATURES")
print("=" * 100)

df_model = df_model.sort_values(["player", "date", "match_id"]).copy()

if "competition" in df_model.columns:
    
    # Current match competition indicators
    df_model["is_champions_league"] = (
        df_model["competition"]
        .astype(str)
        .str.contains("Champions|UCL|CL", case=False, na=False)
        .astype(int)
    )
    
    df_model["is_premier_league"] = (
        df_model["competition"]
        .astype(str)
        .str.contains("Premier", case=False, na=False)
        .astype(int)
    )
    
    df_model["is_cup_match"] = (
        (df_model["is_champions_league"] == 0) &
        (df_model["is_premier_league"] == 0)
    ).astype(int)
    
    # Player's CL minutes in the current row, used only to build shifted history
    df_model["cl_minutes_match"] = (
        df_model["min"] * df_model["is_champions_league"]
    )
    
    # Previous CL exposure, shifted to avoid leakage
    df_model["played_champions_league_last_match"] = (
        df_model
        .groupby("player")["is_champions_league"]
        .shift(1)
        .fillna(0)
    )
    
    df_model["champions_league_minutes_last_3"] = (
        df_model
        .groupby("player")["cl_minutes_match"]
        .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).sum())
        .fillna(0)
    )
    
    df_model["champions_league_minutes_last_5"] = (
        df_model
        .groupby("player")["cl_minutes_match"]
        .transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).sum())
        .fillna(0)
    )
    
    df_model["champions_league_matches_last_3"] = (
        df_model
        .groupby("player")["is_champions_league"]
        .transform(lambda x: x.shift(1).rolling(window=3, min_periods=1).sum())
        .fillna(0)
    )
    
    print("Competition features created:")
    print("  ✓ is_champions_league: current match is Champions League")
    print("  ✓ is_premier_league: current match is Premier League")
    print("  ✓ is_cup_match: current match is domestic cup / other")
    print("  ✓ champions_league_minutes_last_3: previous CL load")
    print("  ✓ champions_league_minutes_last_5: previous CL load")
    print("  ✓ champions_league_matches_last_3: previous CL frequency")
    print("  ✓ played_champions_league_last_match: previous match was CL")
    
    print("\nCompetition distribution:")
    print(df_model["competition"].value_counts().to_string())
    
    print("\nPreview:")
    preview_cols = [
        "player",
        "date",
        "competition",
        "opponent",
        "min",
        "is_champions_league",
        "played_champions_league_last_match",
        "champions_league_minutes_last_3",
        "champions_league_matches_last_3",
        "decline_target"
    ]
    
    print(
        df_model
        .dropna(subset=["rolling_baseline_5"])
        [preview_cols]
        .head(20)
        .round(2)
        .to_string(index=False)
    )

else:
    print("⚠️ Competition column not found in dataset")
    df_model["is_champions_league"] = 0
    df_model["is_premier_league"] = 0
    df_model["is_cup_match"] = 0
    df_model["champions_league_minutes_last_3"] = 0
    df_model["champions_league_minutes_last_5"] = 0
    df_model["champions_league_matches_last_3"] = 0
    df_model["played_champions_league_last_match"] = 0


4. CREATE COMPETITION AND CHAMPIONS LEAGUE CONTEXT FEATURES
Competition features created:
  ✓ is_champions_league: current match is Champions League
  ✓ is_premier_league: current match is Premier League
  ✓ is_cup_match: current match is domestic cup / other
  ✓ champions_league_minutes_last_3: previous CL load
  ✓ champions_league_minutes_last_5: previous CL load
  ✓ champions_league_matches_last_3: previous CL frequency
  ✓ played_champions_league_last_match: previous match was CL

Competition distribution:
competition
Premier League    408
Champions Lg      149
EFL Cup            59
FA Cup             13

Preview:
     player       date    competition           opponent  min  is_champions_league  played_champions_league_last_match  champions_league_minutes_last_3  champions_league_matches_last_3  decline_target
  Ben White 2024-08-31 Premier League           Brighton   90                    0                                 0.0                              0.0                     

## 5. Add Position and Match-Context Features

In [18]:
# ============================================================================
# 5 MATCH CONTEXT FEATURE CHECK
# ============================================================================
# Purpose:
# - Validate match-context features before building the model matrix
# - Avoid manual dummy encoding; categorical variables will be encoded later
#   inside the sklearn pipeline.

print("\n" + "=" * 100)
print("4.1 MATCH CONTEXT FEATURE CHECK")
print("=" * 100)

# ---------------------------------------------------------------------------
# Venue encoding
# ---------------------------------------------------------------------------

if "venue" in df_model.columns:
    df_model["is_away"] = (
        df_model["venue"]
        .astype(str)
        .str.lower()
        .eq("away")
        .astype(int)
    )
    print("✓ is_away created from venue")

    print("\nVenue distribution:")
    print(df_model["venue"].value_counts().to_string())
else:
    df_model["is_away"] = 0
    print("⚠️ venue column not found. is_away set to 0.")

# ---------------------------------------------------------------------------
# Position groups
# ---------------------------------------------------------------------------

if "position_group" in df_model.columns:
    print("\nPosition groups:")
    print(sorted(df_model["position_group"].dropna().unique()))
else:
    print("\n⚠️ position_group not found")

# ---------------------------------------------------------------------------
# Previous performance and decline-history features
# These should already exist from the pre-match workload feature block.
# ---------------------------------------------------------------------------

required_context_features = [
    "position_group",
    "competition",
    "venue",
    "is_away",
    "overall_score_last_1",
    "overall_score_avg_last_3",
    "overall_score_avg_last_5",
    "decline_last_1",
    "declines_last_3",
    "declines_last_5"
]

print("\nContext/history feature availability:")
for col in required_context_features:
    if col in df_model.columns:
        missing = df_model[col].isna().sum()
        print(f"  ✓ {col:32} missing: {missing}")
    else:
        print(f"  ✗ {col:32} MISSING")

print("\nPreview:")
preview_cols = [
    "player",
    "date",
    "opponent",
    "position_group",
    "competition",
    "venue",
    "is_away",
    "overall_score_last_1",
    "decline_last_1",
    "declines_last_3",
    "decline_target"
]

preview_cols = [col for col in preview_cols if col in df_model.columns]

print(
    df_model
    .dropna(subset=["rolling_baseline_5"])
    [preview_cols]
    .head(15)
    .round(2)
    .to_string(index=False)
)


4.1 MATCH CONTEXT FEATURE CHECK
✓ is_away created from venue

Venue distribution:
venue
Home    321
Away    308

Position groups:
['Attacking Midfielder/Winger', 'Central Midfielder', 'Centre-back', 'Forward', 'Full-back']

Context/history feature availability:
  ✓ position_group                   missing: 0
  ✓ competition                      missing: 0
  ✓ venue                            missing: 0
  ✓ is_away                          missing: 0
  ✓ overall_score_last_1             missing: 0
  ✓ overall_score_avg_last_3         missing: 0
  ✓ overall_score_avg_last_5         missing: 0
  ✓ decline_last_1                   missing: 0
  ✓ declines_last_3                  missing: 0
  ✓ declines_last_5                  missing: 0

Preview:
   player       date           opponent position_group    competition venue  is_away  overall_score_last_1  decline_last_1  declines_last_3  decline_target
Ben White 2024-08-31           Brighton      Full-back Premier League  Home        0       

## 6. Build Model-Ready Dataset

In [31]:
# ============================================================================
# 5. BUILD MODEL-READY DATASET
# ============================================================================
# Select predictors using PRE-MATCH information only.
# Do not include current-match performance variables.

print("\n" + "=" * 100)
print("5. BUILD MODEL-READY DATASET")
print("=" * 100)

# Keep only rows with valid player-relative baseline
model_ready = df_model.dropna(subset=["rolling_baseline_5"]).copy()

target_col = "decline_target"

numeric_features = [
    # Rest and workload
    "rest_days",
    "short_rest_binary",
    "moderate_rest_binary",
    "normal_rest_binary",
    "long_gap_return",
    "minutes_last_1",
    "minutes_last_3",
    "minutes_last_5",
    "avg_minutes_last_3",
    "avg_minutes_last_5",
    "starts_last_3",
    "starts_last_5",
    "short_rest_count_last_3",
    "short_rest_count_last_5",

    # Competition / Champions League context
    "is_away",
    "is_champions_league",
    "is_premier_league",
    "is_cup_match",
    "champions_league_minutes_last_3",
    "champions_league_minutes_last_5",
    "champions_league_matches_last_3",
    "played_champions_league_last_match",

    # Previous performance and decline history
    "overall_score_last_1",
    "overall_score_avg_last_3",
    "overall_score_avg_last_5",
    "decline_last_1",
    "declines_last_3",
    "declines_last_5",
]

categorical_features = [
    "position_group",
    "competition",
    "venue",
]

# Keep only available columns
numeric_features = [col for col in numeric_features if col in model_ready.columns]
categorical_features = [col for col in categorical_features if col in model_ready.columns]

# Combine and remove duplicates (maintain order)
predictor_cols = list(dict.fromkeys(numeric_features + categorical_features))

# Leakage check
leakage_cols = [
    "overall_score",
    "attacking_score",
    "attacking_score_normalized",
    "defensive_score",
    "defensive_score_normalized",
    "discipline_score_normalized",
    "performance_vs_baseline",
    "pct_vs_baseline_5",
    "decline_target",
    "min",
    "gls",
    "ast",
    "shots",
    "shots_on_target",
    "crosses",
    "tackles_won",
    "interceptions",
]

leakage_in_predictors = [col for col in leakage_cols if col in predictor_cols]

# Build modelling dataframe
id_cols = [
    "match_id",
    "date",
    "player",
    "opponent",
]

id_cols = [col for col in id_cols if col in model_ready.columns]

model_df = model_ready[id_cols + [target_col] + predictor_cols].copy()

# Drop rows with missing predictors/target
model_df = model_df.dropna(subset=[target_col] + predictor_cols).copy()

X = model_df[predictor_cols].copy()
y = model_df[target_col].copy()

print("MODEL-READY DATASET")
print(f"  Total rows: {len(model_df)}")
print(f"  Total predictors: {len(predictor_cols)}")
print(f"  Numeric predictors: {len(numeric_features)}")
print(f"  Categorical predictors: {len(categorical_features)}")

print("\nTarget distribution:")
print(y.value_counts().sort_index().to_string())
print(f"Decline rate: {y.mean():.1%}")

print("\nLeakage check:")
if leakage_in_predictors:
    print("⚠️ Leakage columns found in predictors:")
    print(leakage_in_predictors)
else:
    print("✓ No obvious leakage columns included")

print(f"\nNumeric predictors ({len(numeric_features)}):")
for col in numeric_features:
    print(f"  • {col}")

print(f"\nCategorical predictors ({len(categorical_features)}):")
for col in categorical_features:
    print(f"  • {col}")

print("\nPreview:")
preview_cols = [
    "date",
    "player",
    "opponent",
    "position_group",
    "competition",
    "venue",
    "decline_target",
    "rest_days",
    "minutes_last_3",
    "starts_last_3",
    "champions_league_minutes_last_3",
    "overall_score_last_1",
]
preview_cols = [col for col in preview_cols if col in model_df.columns]

print(model_df[preview_cols].head(15).round(2).to_string(index=False))


5. BUILD MODEL-READY DATASET
MODEL-READY DATASET
  Total rows: 586
  Total predictors: 31
  Numeric predictors: 28
  Categorical predictors: 3

Target distribution:
decline_target
0    363
1    223
Decline rate: 38.1%

Leakage check:
✓ No obvious leakage columns included

Numeric predictors (28):
  • rest_days
  • short_rest_binary
  • moderate_rest_binary
  • normal_rest_binary
  • long_gap_return
  • minutes_last_1
  • minutes_last_3
  • minutes_last_5
  • avg_minutes_last_3
  • avg_minutes_last_5
  • starts_last_3
  • starts_last_5
  • short_rest_count_last_3
  • short_rest_count_last_5
  • is_away
  • is_champions_league
  • is_premier_league
  • is_cup_match
  • champions_league_minutes_last_3
  • champions_league_minutes_last_5
  • champions_league_matches_last_3
  • played_champions_league_last_match
  • overall_score_last_1
  • overall_score_avg_last_3
  • overall_score_avg_last_5
  • decline_last_1
  • declines_last_3
  • declines_last_5

Categorical predictors (3):
  • posit

## 7. Train/Test Split Using Time-Aware Validation

In [32]:
# ============================================================================
# 6. TIME-AWARE TRAIN / TEST SPLIT
# ============================================================================

print("\n" + "=" * 100)
print("6. TIME-AWARE TRAIN / TEST SPLIT")
print("=" * 100)

# Sort by date to mimic real forecasting
model_df_sorted = model_df.sort_values(["date", "match_id", "player"]).reset_index(drop=True)

# Use 70/30 chronological split
split_idx = int(len(model_df_sorted) * 0.70)

train_df = model_df_sorted.iloc[:split_idx].copy()
test_df = model_df_sorted.iloc[split_idx:].copy()

X_train = train_df[predictor_cols].copy()
y_train = train_df[target_col].copy()

X_test = test_df[predictor_cols].copy()
y_test = test_df[target_col].copy()

print("TIME-AWARE TRAIN/TEST SPLIT")

print("\nTraining set:")
print(f"  Rows: {len(train_df)}")
print(f"  Date range: {train_df['date'].min()} to {train_df['date'].max()}")
print(f"  Decline rate: {y_train.mean():.1%}")

print("\nTest set:")
print(f"  Rows: {len(test_df)}")
print(f"  Date range: {test_df['date'].min()} to {test_df['date'].max()}")
print(f"  Decline rate: {y_test.mean():.1%}")

print("\nFeature types:")
print(f"  Numeric features: {len(numeric_features)}")
print(f"  Categorical features: {len(categorical_features)}")


6. TIME-AWARE TRAIN / TEST SPLIT
TIME-AWARE TRAIN/TEST SPLIT

Training set:
  Rows: 410
  Date range: 2024-08-31 00:00:00 to 2025-03-04 00:00:00
  Decline rate: 38.8%

Test set:
  Rows: 176
  Date range: 2025-03-04 00:00:00 to 2025-05-25 00:00:00
  Decline rate: 36.4%

Feature types:
  Numeric features: 28
  Categorical features: 3


In [33]:
# ============================================================================
# 6.1 PREPROCESSING PIPELINE
# ============================================================================

from sklearn.preprocessing import OneHotEncoder

print("\n" + "=" * 100)
print("6.1 PREPROCESSING PIPELINE")
print("=" * 100)

# Ensure clean categorical feature list
categorical_features_clean = list(dict.fromkeys(categorical_features))

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore", drop=None), categorical_features_clean),
    ],
    remainder="drop"
)

print("✓ Preprocessing pipeline created")
print("  Numeric features will be standardized")
print("  Categorical features will be one-hot encoded")

# Fit preprocessor on training set only
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"\nProcessing summary:")
print(f"  X_train shape: {X_train.shape} → {X_train_processed.shape}")
print(f"  X_test shape:  {X_test.shape} → {X_test_processed.shape}")


6.1 PREPROCESSING PIPELINE
✓ Preprocessing pipeline created
  Numeric features will be standardized
  Categorical features will be one-hot encoded

Processing summary:
  X_train shape: (410, 31) → (410, 39)
  X_test shape:  (176, 31) → (176, 39)


In [23]:
# Diagnostic: Check for duplicate categorical columns
print("\nDiagnostic check:")
print(f"categorical_features: {categorical_features}")
print(f"Duplicates in categorical_features: {[col for col in categorical_features if categorical_features.count(col) > 1]}")
print(f"\nX_train columns sample: {list(X_train.columns[:10])}")
print(f"X_train shape: {X_train.shape}")
print(f"Unique columns in X_train: {len(X_train.columns)}")

# Remove any duplicate categorical features
categorical_features_clean = list(dict.fromkeys(categorical_features))
print(f"\nCleaned categorical_features: {categorical_features_clean}")

# Verify all categorical features are in X_train
for col in categorical_features_clean:
    if col in X_train.columns:
        print(f"  ✓ {col}: {X_train[col].nunique()} unique values")
    else:
        print(f"  ✗ {col}: NOT IN X_train")


Diagnostic check:
categorical_features: ['position_group', 'competition', 'venue']
Duplicates in categorical_features: []

X_train columns sample: ['rest_days', 'short_rest_binary', 'moderate_rest_binary', 'normal_rest_binary', 'long_gap_return', 'minutes_last_1', 'minutes_last_3', 'minutes_last_5', 'avg_minutes_last_3', 'avg_minutes_last_5']
X_train shape: (410, 34)
Unique columns in X_train: 34

Cleaned categorical_features: ['position_group', 'competition', 'venue']
  ✓ position_group: position_group    5
position_group    5
dtype: int64 unique values
  ✓ competition: competition    4
competition    4
dtype: int64 unique values
  ✓ venue: venue    2
venue    2
dtype: int64 unique values


In [29]:
# Check for duplicate column names
print("\nDetailed column check:")
print(f"X_train.columns: {list(X_train.columns)}")
print(f"\nDuplicate column names:")
from collections import Counter
col_counts = Counter(X_train.columns)
duplicates = {col: count for col, count in col_counts.items() if count > 1}
if duplicates:
    print(f"  {duplicates}")
else:
    print(f"  None found")

print(f"\nAll columns in X_train:")
for i, col in enumerate(X_train.columns):
    print(f"  {i:2d}: {col}")


Detailed column check:
X_train.columns: ['rest_days', 'short_rest_binary', 'moderate_rest_binary', 'normal_rest_binary', 'long_gap_return', 'minutes_last_1', 'minutes_last_3', 'minutes_last_5', 'avg_minutes_last_3', 'avg_minutes_last_5', 'starts_last_3', 'starts_last_5', 'short_rest_count_last_3', 'short_rest_count_last_5', 'is_away', 'is_champions_league', 'is_premier_league', 'is_cup_match', 'champions_league_minutes_last_3', 'champions_league_minutes_last_5', 'champions_league_matches_last_3', 'played_champions_league_last_match', 'overall_score_last_1', 'overall_score_avg_last_3', 'overall_score_avg_last_5', 'decline_last_1', 'declines_last_3', 'declines_last_5', 'position_group', 'position_group', 'competition', 'competition', 'venue', 'venue']

Duplicate column names:
  {'position_group': 2, 'competition': 2, 'venue': 2}

All columns in X_train:
   0: rest_days
   1: short_rest_binary
   2: moderate_rest_binary
   3: normal_rest_binary
   4: long_gap_return
   5: minutes_last_1


In [30]:
# Check predictor_cols
print("\npredictor_cols check:")
print(f"Length: {len(predictor_cols)}")
print(f"Unique: {len(set(predictor_cols))}")
print(f"Duplicates: {[col for col in set(predictor_cols) if predictor_cols.count(col) > 1]}")

# Check model_df columns
print(f"\nmodel_df.columns ({len(model_df.columns)} columns):")
print(list(model_df.columns))

# Check if model_df has duplicates
from collections import Counter
col_counts = Counter(model_df.columns)
model_df_dups = {col: count for col, count in col_counts.items() if count > 1}
print(f"\nDuplicate columns in model_df: {model_df_dups if model_df_dups else 'None'}")


predictor_cols check:
Length: 31
Unique: 31
Duplicates: []

model_df.columns (39 columns):
['match_id', 'date', 'player', 'opponent', 'position_group', 'competition', 'venue', 'decline_target', 'rest_days', 'short_rest_binary', 'moderate_rest_binary', 'normal_rest_binary', 'long_gap_return', 'minutes_last_1', 'minutes_last_3', 'minutes_last_5', 'avg_minutes_last_3', 'avg_minutes_last_5', 'starts_last_3', 'starts_last_5', 'short_rest_count_last_3', 'short_rest_count_last_5', 'is_away', 'is_champions_league', 'is_premier_league', 'is_cup_match', 'champions_league_minutes_last_3', 'champions_league_minutes_last_5', 'champions_league_matches_last_3', 'played_champions_league_last_match', 'overall_score_last_1', 'overall_score_avg_last_3', 'overall_score_avg_last_5', 'decline_last_1', 'declines_last_3', 'declines_last_5', 'position_group', 'competition', 'venue']

Duplicate columns in model_df: {'position_group': 2, 'competition': 2, 'venue': 2}


# Baseline: predict majority class (no decline)
baseline_pred = np.zeros_like(y_test)

from sklearn.metrics import accuracy_score

baseline_acc = accuracy_score(y_test, baseline_pred)
baseline_auc = 0.5  # Random classifier AUC

print(f"BASELINE MODEL (predict always: No Decline)")
print(f"  Accuracy: {baseline_acc:.3f}")
print(f"  ROC-AUC: {baseline_auc:.3f}")
print(f"\nThis is the minimum we should beat with a real model.")

## 9. Logistic Regression Model

In [ ]:
# Train logistic regression
lr_model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_train = lr_model.predict(X_train_scaled)
y_pred_test = lr_model.predict(X_test_scaled)
y_pred_proba_train = lr_model.predict_proba(X_train_scaled)[:, 1]
y_pred_proba_test = lr_model.predict_proba(X_test_scaled)[:, 1]

print(f"LOGISTIC REGRESSION MODEL TRAINED")
print(f"  Predictors: {len(predictor_cols)}")
print(f"  Regularization: L2 (default)")
print(f"  Class weight: Balanced (accounts for decline rarity)")

## 10. Model Evaluation

In [ ]:
# Evaluation metrics
train_acc = accuracy_score(y_train, y_pred_train)
test_acc = accuracy_score(y_test, y_pred_test)
train_auc = roc_auc_score(y_train, y_pred_proba_train)
test_auc = roc_auc_score(y_test, y_pred_proba_test)
train_f1 = f1_score(y_train, y_pred_train)
test_f1 = f1_score(y_test, y_pred_test)

print(f"MODEL PERFORMANCE")
print(f"\nTraining Set:")
print(f"  Accuracy: {train_acc:.3f}")
print(f"  ROC-AUC:  {train_auc:.3f}")
print(f"  F1-Score: {train_f1:.3f}")

print(f"\nTest Set:")
print(f"  Accuracy: {test_acc:.3f}")
print(f"  ROC-AUC:  {test_auc:.3f}")
print(f"  F1-Score: {test_f1:.3f}")

print(f"\nImprovement over baseline:")
print(f"  Accuracy: {test_acc - baseline_acc:+.3f}")
print(f"  ROC-AUC:  {test_auc - baseline_auc:+.3f}")

# Detailed classification report
print(f"\nDetailed Test Set Classification Report:")
print(classification_report(y_test, y_pred_test, target_names=['No Decline', 'Decline']))

# Model coefficients (on standardized scale)
coef_df = pd.DataFrame({
    'Feature': predictor_cols,
    'Coefficient': lr_model.coef_[0],
    'Abs_Coef': np.abs(lr_model.coef_[0])
}).sort_values('Abs_Coef', ascending=False)

# Map coefficients to odds ratios
coef_df['Odds_Ratio'] = np.exp(coef_df['Coefficient'])
coef_df['Risk_Direction'] = coef_df['Coefficient'].apply(
    lambda x: 'Increases Risk' if x > 0 else 'Decreases Risk'
)

print(f"MODEL COEFFICIENTS (Feature Importance)\n")
print(coef_df[['Feature', 'Coefficient', 'Odds_Ratio', 'Risk_Direction']].to_string(index=False))

print(f"\nInterpretation (example):")
print(f"  If short_rest_binary coef = +0.45 → Odds Ratio = {np.exp(0.45):.2f}")
print(f"  Meaning: Each 1-unit increase in short_rest increases decline odds by 57%")
print(f"\nTop 5 Risk Factors:")
top_risk = coef_df[coef_df['Coefficient'] > 0].head(5)
for idx, row in top_risk.iterrows():
    print(f"  {row['Feature']:35} → +{row['Coefficient']:.3f} (↑ {(row['Odds_Ratio']-1)*100:.1f}% risk)")

# Add risk predictions to test set
test_df['decline_probability'] = y_pred_proba_test
test_df['predicted_decline'] = y_pred_test
test_df['actual_decline'] = y_test.values

# Create aggregated player-level summary for next matches
player_risk_summary = test_df.groupby('player').agg({
    'decline_probability': ['mean', 'std', 'count'],
    'actual_decline': 'sum',
    'position_group': 'first'
}).round(3)

player_risk_summary.columns = ['Avg_Risk', 'Risk_Std', 'Matches', 'Actual_Declines', 'Position']
player_risk_summary['Decline_Rate'] = (player_risk_summary['Actual_Declines'] / player_risk_summary['Matches']).round(3)
player_risk_summary = player_risk_summary.sort_values('Avg_Risk', ascending=False)

print(f"TRAINER-FACING RISK TABLE (Test Set)")
print(f"\nPlayer Risk Summary (sorted by average decline risk):")
print(f"  Avg_Risk: Average predicted decline probability")
print(f"  Risk_Std: Variability in risk (consistent vs unpredictable)")
print(f"  Matches: Number of test set appearances")
print(f"  Actual_Declines: Observed declines in test set")
print(f"  Decline_Rate: Actual decline frequency\n")
print(player_risk_summary.head(15).to_string())

print(f"\nKey Insights for Coaching Staff:")
high_risk = player_risk_summary[player_risk_summary['Avg_Risk'] > 0.35]
print(f"  High-risk players (>35% decline probability): {len(high_risk)} players")
if len(high_risk) > 0:
    print(f"  {', '.join(high_risk.index.tolist())}")
    print(f"  → Consider: Extra recovery, rotation, workload management")

# Save outputs
model_metadata = {
    'Model Type': 'Logistic Regression',
    'Training Set Accuracy': train_acc,
    'Test Set Accuracy': test_acc,
    'Test Set ROC-AUC': test_auc,
    'Number of Predictors': len(predictor_cols),
    'Training Samples': len(train_df),
    'Test Samples': len(test_df),
    'No-Leakage Rule': 'Only pre-match predictors used'
}

# Save player risk summary
risk_table_file = OUTPUTS_PATH / f'{TEAM_SLUG}_player_decline_risk_table.csv'
player_risk_summary.to_csv(risk_table_file)

# Save coefficients
coef_file = OUTPUTS_PATH / f'{TEAM_SLUG}_model_coefficients.csv'
coef_df.to_csv(coef_file, index=False)

# Save test predictions (for validation)
test_pred_file = OUTPUTS_PATH / f'{TEAM_SLUG}_test_predictions.csv'
test_df.to_csv(test_pred_file, index=False)

print(f"\n" + "="*100)
print(f"NOTEBOOK 03 OUTPUTS SAVED")
print(f"="*100)
print(f"\n✓ Player Risk Table:      {risk_table_file.name}")
print(f"✓ Model Coefficients:     {coef_file.name}")
print(f"✓ Test Set Predictions:   {test_pred_file.name}")
print(f"\nModel Performance Summary:")
for key, val in model_metadata.items():
    if isinstance(val, float):
        print(f"  {key:30} {val:.3f}")
    else:
        print(f"  {key:30} {val}")

print(f"\n" + "="*100)
print(f"LIMITATIONS AND NEXT STEPS")
print(f"="*100)

limitations = f"""
## Model Limitations

1. **Sample Size**: Only {len(train_df)} training samples across {train_df['player'].nunique()} players
   → Risk estimates have wide confidence intervals
   → Rare player-condition combinations may be unreliable

2. **Short Time Window**: Single season (2024/25)
   → Cannot capture long-term fatigue trends
   → Player availability may change next season

3. **Position Aggregation**: {df['position_group'].nunique()} position groups
   → May hide role-specific fatigue patterns (e.g., center-back vs fullback)
   → Consider separate models per position for future work

4. **Missing Variables**:
   → Opponent strength not included (influence on fatigue?)
   → Player age/experience not modeled
   → Injury history unavailable
   → Team formation/tactics not considered

5. **Decline Definition**: 10% drop below player baseline
   → Arbitrary threshold (could be 5%, 15%, position-specific)
   → Recommend clinical review with medical staff

## Recommended Next Steps (Future Work)

1. **Data Expansion**:
   - Collect 2-3 seasons (2022-2025) for stability
   - Add opponent strength, player age, injury flags
   - Separate models per position

2. **Model Improvements**:
   - Hyperparameter tuning (regularization strength)
   - Calibration (ensure risk probabilities are well-calibrated)
   - Ensemble methods (Random Forest, Gradient Boosting)
   - Time-series cross-validation (rolling windows)

3. **Clinical Validation**:
   - Work with medical staff to define "meaningful" decline
   - Validate against injury rates, muscle strains
   - Test thresholds for actionable interventions

4. **Deployment**:
   - Build interactive dashboard for coaching staff
   - Integrate with match-planning tool
   - Monthly retraining with new match data
   - A/B test impact on team rotation decisions

## Interpretation for Coaching Staff

✓ **What this model does**:
  • Predicts player performance decline risk based on fatigue indicators
  • Identifies high-risk players (>35% probability) for extra monitoring
  • Shows which factors increase risk (rest, competition load, position)

✗ **What this model does NOT do**:
  • Predict injuries (this is performance decline, not injury)
  • Replace medical staff assessment (complement only)
  • Guarantee specific players will decline (probabilities, not certainties)
  • Account for tactical adjustments, form trends, morale

💡 **Recommended Usage**:
  1. Review risk table before team selection
  2. Flag high-risk players (>35%) for discussion with medical staff
  3. Consider rotation options for consistent high-risk players
  4. Monitor predictions vs. actual performance (validation)
  5. Refine threshold (35%?) based on actual outcomes
"""

print(limitations)